# Sentinel-5P TROPOMI NO2

Fetch a Sentinel-5P TROPOMI Level 2 NO2 product (atmospheric composition) over a region of interest.

This notebook performs a **real** EUMETSAT Data Store download when `EUMETSAT_CONSUMER_KEY` / `EUMETSAT_CONSUMER_SECRET` are set (see the [authentication guide](../../reference/eumetsat/authentication.md)). Without credentials it prints a `skipped live query` line and still runs to the end.

## Setup

Import `EarthLens` (the unified entry point) and prepare a local output directory for the downloaded products.

In [ ]:
from pathlib import Path

from earthlens.core import EarthLens

OUT_DIR = Path('eumetsat_output')
OUT_DIR.mkdir(exist_ok=True)

## Download the NO2 product

Describe the request first: the EUMETSAT backend, the `s5p-l2-no2` dataset, the NO2 variable, a one-day window, and a bounding box over central Europe. Writing the output to `OUT_DIR`.

In [ ]:
request = EarthLens(
    data_source='eumetsat',
    dataset='s5p-l2-no2',
    variables=['L2__NO2___'],
    start='2024-06-01',
    end='2024-06-01',
    aoi=[0.0, 40.0, 15.0, 55.0],
    path=str(OUT_DIR),
)

Run the download. The call is wrapped in a `try`/`except` so the notebook still runs end-to-end without credentials: if `EUMETSAT_CONSUMER_KEY` / `EUMETSAT_CONSUMER_SECRET` are missing (or the Data Store has a transient hiccup), it prints a `skipped live query` line instead of failing.

In [ ]:
paths = None
try:
    paths = request.download(progress_bar=False)
    print(len(paths), 'product(s):', [Path(p).name for p in paths])
except Exception as exc:
    # No EUMETSAT_CONSUMER_KEY/SECRET, or a transient Data Store hiccup:
    # the notebook still runs top-to-bottom and reports the skip.
    print(f'skipped live query: {type(exc).__name__}: {exc}')

## Reading the product

S5P L2 products are NetCDF swaths — open the downloaded file with xarray (group `PRODUCT`) to read the tropospheric NO2 column.